In [ ]:
%pip install datasets pillow pandas pyarrow

   ---------------------------------------- 0.0/604.4 kB ? eta -:--:--
   ---------------------------------------- 604.4/604.4 kB 7.3 MB/s  0:00:00
   ---------------------------------------- 0.0/838.0 kB ? eta -:--:--
   ---------------------------------------- 838.0/838.0 kB 9.3 MB/s  0:00:00
   ---------------------------------------- 0.0/3.8 MB ? eta -:--:--
   --------------------- ------------------ 2.1/3.8 MB 11.8 MB/s eta 0:00:01
   ---------------------------------------- 3.8/3.8 MB 10.9 MB/s  0:00:00
   ---------------------------------------- 0.0/27.9 MB ? eta -:--:--
   --- ------------------------------------ 2.4/27.9 MB 11.2 MB/s eta 0:00:03
   ------ --------------------------------- 4.7/27.9 MB 11.0 MB/s eta 0:00:03
   ---------- ----------------------------- 7.1/27.9 MB 11.2 MB/s eta 0:00:02
   ------------- -------------------------- 9.4/27.9 MB 11.3 MB/s eta 0:00:02
   ---------------- ----------------------- 11.8/27.9 MB 11.2 MB/s eta 0:00:02
   --------------------

In [4]:
%pip install requests


   ---------------------------------------- 0/4 [urllib3]
   ---------------------------------------- 0/4 [urllib3]
   ---------- ----------------------------- 1/4 [charset_normalizer]
   -------------------- ------------------- 2/4 [certifi]
   ------------------------------ --------- 3/4 [requests]
   ---------------------------------------- 4/4 [requests]

Note: you may need to restart the kernel to use updated packages.


In [8]:
# Install: pip install datasets
from datasets import load_dataset
import requests
from PIL import Image
import pandas as pd
from pathlib import Path
 
# Load dataset from HuggingFace
print("Loading product dataset...")
try:
    # Try loading the dataset
    dataset = load_dataset("ashraq/fashion-product-images-small", split="train[:100]")  # First 100 samples
    print(f"✓ Loaded {len(dataset)} products")
    
    # Convert to pandas for easier manipulation
    products_df = pd.DataFrame(dataset)
    print(f"Dataset columns: {products_df.columns.tolist()}")
    
except Exception as e:
    print(f"⚠ Could not load HuggingFace dataset: {e}")
    print("Using local images instead...")
    
    # Alternative: Use local images
    # Create a products.json file with product information
    products_data = [
        {
            "id": 1,
            "name": "Wireless Headphones",
            "price": 79.99,
            "category": "Electronics",
            "image_path": "images/product1.jpg"
        },
        # Add more products...
    ]
    
    products_df = pd.DataFrame(products_data)
 
# Create images directory
images_dir = Path("product_images")
images_dir.mkdir(exist_ok=True)
 
print(f"\n✓ Dataset prepared!")
print(f"  Total products: {len(products_df)}")

Loading product dataset...
✓ Loaded 100 products
Dataset columns: ['id', 'gender', 'masterCategory', 'subCategory', 'articleType', 'baseColour', 'season', 'year', 'usage', 'productDisplayName', 'image']

✓ Dataset prepared!
  Total products: 100


In [9]:
products_df.head()

,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,<PIL.JpegImagePlugin.JpegImageFile image mode=...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,<PIL.JpegImagePlugin.JpegImageFile image mode=...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch,<PIL.Image.Image image mode=L size=60x80 at 0x...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants,<PIL.JpegImagePlugin.JpegImageFile image mode=...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,<PIL.Image.Image image mode=RGB size=60x80 at ...


In [10]:
import base64
from io import BytesIO

def encode_image_to_base64(image):
    """Encode a PIL image to a base64 string."""
    buffer = BytesIO()
    image.save(buffer, format="JPEG")
    encoded = base64.b64encode(buffer.getvalue()).decode("utf-8")
    return encoded

# Example usage
sample_image = products_df.iloc[0]["image"]

encoded_image = encode_image_to_base64(sample_image)

print(f"Encoded image length: {len(encoded_image)} characters")
print(f"Encoded prefix: {encoded_image[:40]}...")

Encoded image length: 2388 characters
Encoded prefix: /9j/4AAQSkZJRgABAQAAAQABAAD/2wBDAAgGBgcG...


In [11]:
# Placeholder prices based on product category
placeholder_prices = {
    "Apparel": 29.99,
    "Accessories": 24.99,
    "Footwear": 59.99,
    "Personal Care": 14.99
}

def create_product_listing_prompt(product_name, price, category, additional_info=None):
    """
    Create a prompt for generating product listings.

    Parameters:
    - product_name: Name of the product
    - price: Price of the product
    - category: Product category
    - additional_info: Optional additional information

    Returns:
    - Formatted prompt string
    """

    prompt = f"""You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive language
   - Include relevant details visible in the image
3. **Key Features** (bullet points, 5-7 items)
4. **SEO Keywords** (comma-separated, 10-15 relevant keywords)

Format your response as JSON with the following structure:
{{
    "title": "Product title here",
    "description": "Full description here",
    "features": ["Feature 1", "Feature 2"],
    "keywords": "keyword1, keyword2, ..."
}}

Be specific about what you see in the image. Mention colors, materials, design elements, and any distinctive features. Do not invent details that cannot be verified from the image or product information."""

    return prompt


# Test prompt creation using HuggingFace dataset
sample_product = products_df.iloc[0]

# Assign placeholder price based on category
category = sample_product["masterCategory"]
price = placeholder_prices.get(category, 19.99)

test_prompt = create_product_listing_prompt(
    product_name=sample_product["productDisplayName"],
    price=price,
    category=category,
    additional_info=f"Color: {sample_product['baseColour']}, Type: {sample_product['articleType']}"
)

print("\n" + "=" * 50)
print("PROMPT TEMPLATE")
print("=" * 50)
print(test_prompt[:500] + "...")


PROMPT TEMPLATE
You are an expert e-commerce copywriter. Analyze the product image and create a compelling product listing.

Product Information:
- Name: Turtle Check Men Navy Blue Shirt
- Price: $29.99
- Category: Apparel
- Additional Info: Color: Navy Blue, Type: Shirts

Please create a professional product listing that includes:

1. **Product Title** (catchy, SEO-friendly, 60 characters max)
2. **Product Description** (detailed, 150-200 words)
   - Highlight key features and benefits
   - Use persuasive lang...


In [5]:
!pip install openai pydantic python-dotenv

In [12]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from pydantic import BaseModel
from typing import List

# Load OPENAI_API_KEY from the .env file next to this notebook
load_dotenv()

# Initialize client (reads OPENAI_API_KEY from the environment)
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [13]:
import json

# Call ChatGPT API with product image and prompt
response = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": test_prompt
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{encoded_image}"
                    }
                }
            ]
        }
    ]
)

# Extract response text
response_text = response.choices[0].message.content

# Remove JSON markdown markers if present
response_text = response_text.strip()

if response_text.startswith("```json"):
    response_text = response_text.removeprefix("```json").strip()

if response_text.startswith("```"):
    response_text = response_text.removeprefix("```").strip()

if response_text.endswith("```"):
    response_text = response_text.removesuffix("```").strip()

# Parse JSON response into Python dictionary
product_listing = json.loads(response_text)

# Display generated product listing
print(json.dumps(product_listing, indent=2, ensure_ascii=False))

{
  "title": "Casual Navy Blue Turtle Check Men's Shirt - Stylish & Versatile",
  "description": "Elevate your wardrobe with the Turtle Check Men Navy Blue Shirt, a perfect blend of style and comfort. Crafted from high-quality fabric, this navy blue short-sleeve shirt features a classic check pattern that adds a touch of modern sophistication to your attire. Its relaxed fit and breathable material make it ideal for casual outings or relaxed office wear. Pair it with jeans or chinos for an effortlessly stylish look that transitions from day to night. The shirt is designed to keep you comfortable while ensuring you stand out in any crowd. Upgrade your collection with this essential piece that combines functionality with contemporary design.",
  "features": [
    "Made from high-quality, breathable fabric",
    "Stylish navy blue color with a classic check pattern",
    "Short sleeves for a relaxed, casual look",
    "Perfect for both formal and casual occasions",
    "Easy to pair with j

In [14]:
import time

# Store generated product listings
generated_listings = []

# API prices per 1 million tokens (GPT-4o-mini)
input_price = 0.15
output_price = 0.60

# Initialize cost tracking
total_cost = 0
total_input_tokens = 0
total_output_tokens = 0
cost_records = []

# Process the first 5 products
for index, product in products_df.head(5).iterrows():
    try:
        print(f"Processing product {index + 1}: {product['productDisplayName']}")

        # Get product metadata
        category = product["masterCategory"]
        price = placeholder_prices.get(category, 19.99)

        # Encode product image to base64
        base64_image = encode_image_to_base64(product["image"])

        # Create product listing prompt
        prompt = create_product_listing_prompt(
            product_name=product["productDisplayName"],
            price=price,
            category=category,
            additional_info=f"Color: {product['baseColour']}, Type: {product['articleType']}"
        )

        # Call ChatGPT API
        response = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {
                    "role": "user",
                    "content": [
                        {
                            "type": "text",
                            "text": prompt
                        },
                        {
                            "type": "image_url",
                            "image_url": {
                                "url": f"data:image/jpeg;base64,{base64_image}"
                            }
                        }
                    ]
                }
            ]
        )

        # Track tokens and calculate API costs
        input_tokens = response.usage.prompt_tokens
        output_tokens = response.usage.completion_tokens

        request_cost = (
            (input_tokens / 1_000_000) * input_price
            + (output_tokens / 1_000_000) * output_price
        )

        total_input_tokens += input_tokens
        total_output_tokens += output_tokens
        total_cost += request_cost

        cost_records.append({
            "product_name": product["productDisplayName"],
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "cost_usd": request_cost
        })

        print(f"Tokens used: {input_tokens + output_tokens}")
        print(f"Request cost: ${request_cost:.6f}")

        # Extract and clean response
        response_text = response.choices[0].message.content.strip()

        if response_text.startswith("```json"):
            response_text = response_text.removeprefix("```json").strip()

        if response_text.startswith("```"):
            response_text = response_text.removeprefix("```").strip()

        if response_text.endswith("```"):
            response_text = response_text.removesuffix("```").strip()

        # Parse JSON response
        product_listing = json.loads(response_text)

        # Save product information and generated listing
        generated_listings.append({
            "id": int(product["id"]),
            "product_name": product["productDisplayName"],
            "category": category,
            "price": price,
            "status": "success",
            "listing": product_listing
        })

        print("✓ Listing generated successfully!")

    except Exception as e:
        print(f"Error processing product: {e}")

        generated_listings.append({
            "id": int(product["id"]),
            "product_name": product["productDisplayName"],
            "status": "failed",
            "error": str(e)
        })

    # Save results after each product
    with open("generated_listings.json", "w", encoding="utf-8") as f:
        json.dump(generated_listings, f, indent=2, ensure_ascii=False)

    # Wait between API calls
    if index != products_df.head(5).index[-1]:
        time.sleep(1)

print("\nBatch processing completed!")
print(f"Total products processed: {len(generated_listings)}")
print(f"Successful listings: {sum(item['status'] == 'success' for item in generated_listings)}")
print(f"Failed listings: {sum(item['status'] == 'failed' for item in generated_listings)}")
print("Results saved to generated_listings.json")

# Save cost tracking results
with open("cost_tracking.json", "w", encoding="utf-8") as f:
    json.dump(cost_records, f, indent=2)

# Display total usage and costs
print(f"Total input tokens: {total_input_tokens}")
print(f"Total output tokens: {total_output_tokens}")
print(f"Total API costs: ${total_cost:.6f}")

# Estimate costs for larger batches
if cost_records:
    average_cost = total_cost / len(cost_records)

    print(f"Estimated cost for 100 products: ${average_cost * 100:.4f}")
    print(f"Estimated cost for 1000 products: ${average_cost * 1000:.4f}")

Processing product 1: Turtle Check Men Navy Blue Shirt
Tokens used: 9038
Request cost: $0.001481
✓ Listing generated successfully!
Processing product 2: Peter England Men Party Blue Jeans
Tokens used: 9019
Request cost: $0.001470
✓ Listing generated successfully!
Processing product 3: Titan Women Silver Watch
Tokens used: 9034
Request cost: $0.001480
✓ Listing generated successfully!
Processing product 4: Manchester United Men Solid Black Track Pants
Tokens used: 9042
Request cost: $0.001483
✓ Listing generated successfully!
Processing product 5: Puma Men Grey T-shirt
Tokens used: 9045
Request cost: $0.001485
✓ Listing generated successfully!

Batch processing completed!
Total products processed: 5
Successful listings: 5
Failed listings: 0
Results saved to generated_listings.json
Total input tokens: 43796
Total output tokens: 1382
Total API costs: $0.007399
Estimated cost for 100 products: $0.1480
Estimated cost for 1000 products: $1.4797


In [15]:
# Create a shorter prompt for cost optimization
def create_optimized_prompt(product_name, price, category, additional_info=None):
    prompt = f"""Analyze the product image and create a professional e-commerce listing.

Product:
- Name: {product_name}
- Price: ${price:.2f}
- Category: {category}
{f'- Additional Info: {additional_info}' if additional_info else ''}

Return valid JSON with:
- "title": Catchy, SEO-friendly, max 60 characters
- "description": 150-200 words highlighting visible features and benefits
- "features": List of 5-7 key features
- "keywords": String with 10-15 comma-separated SEO keywords

Use persuasive language. Describe visible colors and design details. Do not invent unverified product properties.
"""
    return prompt


# Prepare both prompts for the same product
sample_product = products_df.iloc[0]

category = sample_product["masterCategory"]
price = placeholder_prices.get(category, 19.99)

additional_info = f"Color: {sample_product['baseColour']}, Type: {sample_product['articleType']}"

original_prompt = create_product_listing_prompt(
    product_name=sample_product["productDisplayName"],
    price=price,
    category=category,
    additional_info=additional_info
)

optimized_prompt = create_optimized_prompt(
    product_name=sample_product["productDisplayName"],
    price=price,
    category=category,
    additional_info=additional_info
)

print(f"Original prompt length: {len(original_prompt)} characters")
print(f"Optimized prompt length: {len(optimized_prompt)} characters")
print(f"Character reduction: {(1 - len(optimized_prompt) / len(original_prompt)) * 100:.2f}%")

Original prompt length: 1099 characters
Optimized prompt length: 574 characters
Character reduction: 47.77%


In [16]:
# Test optimized prompt with the first product
response_optimized = client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {
            "role": "user",
            "content": [
                {
                    "type": "text",
                    "text": optimized_prompt
                },
                {
                    "type": "image_url",
                    "image_url": {
                        "url": f"data:image/jpeg;base64,{encoded_image}"
                    }
                }
            ]
        }
    ]
)

# Extract token usage
optimized_input_tokens = response_optimized.usage.prompt_tokens
optimized_output_tokens = response_optimized.usage.completion_tokens

# Calculate optimized API costs
optimized_cost = (
    (optimized_input_tokens / 1_000_000) * input_price
    + (optimized_output_tokens / 1_000_000) * output_price
)

# Compare with original API request
original_input_tokens = cost_records[0]["input_tokens"]
original_output_tokens = cost_records[0]["output_tokens"]
original_cost = cost_records[0]["cost_usd"]

print("COST COMPARISON")
print("=" * 50)
print(f"Original input tokens: {original_input_tokens}")
print(f"Optimized input tokens: {optimized_input_tokens}")
print(f"Original output tokens: {original_output_tokens}")
print(f"Optimized output tokens: {optimized_output_tokens}")
print(f"Original cost: ${original_cost:.6f}")
print(f"Optimized cost: ${optimized_cost:.6f}")
print(f"Cost reduction: {(1 - optimized_cost / original_cost) * 100:.2f}%")

COST COMPARISON
Original input tokens: 8760
Optimized input tokens: 8639
Original output tokens: 278
Optimized output tokens: 283
Original cost: $0.001481
Optimized cost: $0.001466
Cost reduction: 1.02%


In [17]:
print(response_optimized.choices[0].message.content)

```json
{
  "title": "Stylish Navy Blue Turtle Check Men’s Shirt",
  "description": "Elevate your wardrobe with the Turtle Check Men Navy Blue Shirt, a perfect blend of style and comfort. This shirt features a classic check pattern, ideal for both casual outings and semi-formal occasions. The rich navy blue color is both timeless and versatile, allowing you to pair it effortlessly with jeans, chinos, or shorts. Crafted from high-quality fabric, this shirt ensures breathability and ease of movement, making it a great choice for all-day wear. The short sleeves add a relaxed vibe, while the tailored fit provides a polished look. Whether you're heading to a brunch or a night out, this shirt will make you stand out with its charming design and modern appeal.",
  "features": [
    "Classic navy blue check pattern",
    "Breathable, high-quality fabric",
    "Short sleeves for a relaxed feel",
    "Tailored fit for a polished look",
    "Versatile for various occasions",
    "Easy to pair wit